In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "MOP"
df = pd.read_excel(file_path, sheet_name=sheet_name)

# ========= 2) 参数模板 =========
params = {
    "weights": np.array([0.5, 0.5]),     # np.ndarray: 各目标权重（建议和为1）
    "x0": np.array([1.0, 1.0]),          # 初始解
    "bounds": ((0, 10), (0, 10))         # 边界
}

def f1(x): return (x[0]-1)**2 + x[1]**2
def f2(x): return x[0]**2 + (x[1]-2)**2

def weighted_obj(x):
    return params["weights"][0]*f1(x) + params["weights"][1]*f2(x)

res = minimize(weighted_obj, x0=params["x0"], method="SLSQP", bounds=params["bounds"])
print(res.x, res.fun)


In [ ]:
"""
多目标规划模型

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "多目标规划模型.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
INITIAL_X = [0.0, 0.0]  # TODO: 请填写[初始解]，说明：长度等于变量数。
BOUNDS = [(-5, 5), (-5, 5)]  # TODO: 请填写[变量边界]，说明：每个变量一个范围。
WEIGHT_GRID_COUNT = 11  # TODO: 请填写[权重网格数]，说明：越大 Pareto 近似越细。
TARGET_VECTOR_1 = np.array([1.0, 2.0])  # TODO: 请填写[目标1参数]，说明：替换为真实问题参数。
TARGET_VECTOR_2 = np.array([-1.0, 1.0])  # TODO: 请填写[目标2参数]，说明：替换为真实问题参数。



REQUIRES_DATA = False  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def f1(x):
    return np.sum((x - TARGET_VECTOR_1) ** 2)


def f2(x):
    return np.sum((x - TARGET_VECTOR_2) ** 2)


def run_model(data: pd.DataFrame) -> None:
    rows = []
    for w in np.linspace(0, 1, WEIGHT_GRID_COUNT):
        obj = lambda x, w=w: w * f1(x) + (1 - w) * f2(x)
        res = minimize(obj, x0=np.array(INITIAL_X, dtype=float), bounds=BOUNDS, method="SLSQP")
        rows.append({"权重f1": w, "f1": f1(res.x), "f2": f2(res.x), "解": res.x.tolist()})
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
